In [1]:
import re
import os

import torch
from unsloth import FastLanguageModel
from datasets import load_dataset, Dataset
from trl import GRPOTrainer, GRPOConfig

[unsloth.import_fixes|WARNING]Unsloth: torch==2.12.0.dev20260223+cu128 requires torchvision>=0.27.0, but found torchvision==0.26.0.dev20260223+cu128. Please refer to https://pytorch.org/get-started/previous-versions/ for more information.
Detected a pre-release build. Continuing with a warning. Set UNSLOTH_SKIP_TORCHVISION_CHECK=1 to silence this.


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
WARNING 03-01 07:06:07 [__init__.py:144] xccl is not enabled in this torch build, communication is not available.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [7]:

# ── Precision toggle ──────────────────────────────────────────────────────────
# RTX 5070 Ti (Blackwell) has native FP8 tensor cores → best speed + VRAM ratio.
# Set LOAD_IN_FP8=True for RTX 40/50 series, or LOAD_IN_4BIT=True for any GPU.
LOAD_IN_FP8  = False    # ← recommended for RTX 40/50; ~60% less VRAM, 1.4× faster vLLM
LOAD_IN_4BIT = True   # ← universal fallback; ~1-2% accuracy tradeoff, works on all GPUs

# ── Training config ───────────────────────────────────────────────────────────
MODEL_NAME  = "HuggingFaceTB/SmolLM-135M-Instruct"
G           = 16       # completions per prompt (GRPO group size)
MAX_NEW     = 8192     # max new tokens per completion
LR          = 1e-4
EPOCHS      = 1
CLIP_EPS    = 0.2      # PPO clip epsilon
TEMPERATURE = 0.9
TRAIN_SIZE  = 500
EVAL_SIZE   = 50

LORA_R      = 8        # LoRA rank (higher = more capacity, more VRAM)
LORA_ALPHA  = 16
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"]

os.environ["UNSLOTH_VLLM_STANDBY"] = "0"  # saves 30%+ VRAM during RL generation

# Unsloth's vLLM standby mode (CuMemAllocator) is incompatible with
# expandable_segments. Override any value PyTorch may have set automatically.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = ""

# Note: VLLM_USE_FLASHINFER_SAMPLER is controlled by unsloth internally
# (set to "1" for GPUs with compute capability >= 8). This requires
# CUDA toolkit 12.8+ so that nvcc can compile for sm_120a (Blackwell).

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | FP8={LOAD_IN_FP8} | 4-bit={LOAD_IN_4BIT}")


Device: cuda | FP8=False | 4-bit=True


In [3]:
# ── Dataset helpers ───────────────────────────────────────────────────────────
def extract_answer(text: str) -> str | None:
    """Extract the final numeric answer from a GSM8K solution string.
    GSM8K solutions end with '#### <number>'."""
    match = re.search(r"####\s*([\d,\-]+)", text)
    if match:
        return match.group(1).replace(",", "").strip()
    return None


SYSTEM_PROMPT = (
    "You are a helpful math tutor. Solve problems step by step. "
    "At the end, write your final answer after '####'."
)


def build_prompt(question: str) -> list[dict]:
    """Return a chat messages list for a GSM8K question."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]


In [4]:
# ── Dataset ───────────────────────────────────────────────────────────────────
# GRPOTrainer expects a HuggingFace Dataset with a "prompt" column (list of
# {"role","content"} dicts) and any extra columns used by the reward function.
print("Loading GSM8K dataset...")
raw = load_dataset("openai/gsm8k", "main")


def make_hf_dataset(split: str, size: int) -> Dataset:
    rows = []
    for ex in raw[split].select(range(size)):
        ans = extract_answer(ex["answer"])
        if ans is not None:
            rows.append({"prompt": build_prompt(ex["question"]), "answer": ans})
    return Dataset.from_list(rows)


train_ds = make_hf_dataset("train", TRAIN_SIZE)
eval_ds  = make_hf_dataset("test",  EVAL_SIZE)
print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")


Loading GSM8K dataset...
Train: 500 | Eval: 50


In [5]:
# ── Reward function ───────────────────────────────────────────────────────────
# TRL calls this as: reward_fn(completions=List[str], answer=List[str], **kwargs)
# It must return a List[float] of the same length.
def reward_fn(completions: list[str], *, answer: list[str], **kwargs) -> list[float]:
    """Rubric reward for GSM8K.

    Correctness:
      +1.00  exact match on the '#### <n>' final answer
      +0.50  gold number appears as a whole word anywhere in the completion

    Format / structure:
      +0.20  uses '#### <answer>' format (even if answer is wrong)
      +0.15  shows arithmetic work: expression like '3 * 4' or '12 / 3'
      +0.15  shows equation results: writes '= <number>' at least once
      +0.10  answer within 2× of the gold value (order-of-magnitude credit)
      +0.05  has 3+ distinct numeric values (multi-step reasoning indicator)
      +0.05  contains any digit (minimum partial credit)

    Penalty:
      -0.20 max  length penalty scales linearly with word count up to MAX_NEW
    """
    rewards = []
    for completion, gold_answer in zip(completions, answer):
        points = 0.0
        pred = extract_answer(completion)

        # ── Correctness ──────────────────────────────────────────────────────
        if pred is not None and pred == gold_answer:
            points += 1.0
        if re.search(rf"\b{re.escape(gold_answer)}\b", completion):
            points += 0.5

        # ── Format / structure ───────────────────────────────────────────────
        if pred is not None:
            points += 0.2
        if re.search(r"\d+\s*[+\-*/×÷]\s*\d+", completion):
            points += 0.15
        if re.search(r"=\s*\$?\s*\d+", completion):
            points += 0.15

        try:
            gold_val = float(gold_answer)
            pred_val = float(pred) if pred is not None else None
            if pred_val is not None and gold_val != 0 and 0.5 <= pred_val / gold_val <= 2.0:
                points += 0.10
        except (ValueError, ZeroDivisionError):
            pass

        distinct_nums = set(re.findall(r"\b\d+(?:\.\d+)?\b", completion))
        if len(distinct_nums) >= 3:
            points += 0.05

        if re.search(r"\d", completion):
            points += 0.05

        # ── Length penalty ────────────────────────────────────────────────────
        num_words = len(completion.split())
        length_penalty = 0.2 * (num_words / MAX_NEW)

        rewards.append(max(0.0, min(points, 1.0) - length_penalty))
    return rewards


print("Reward function defined.")


Reward function defined.


In [9]:
# ── Model ─────────────────────────────────────────────────────────────────────
# FastLanguageModel loads ONE model — no separate ref_model needed.
# GRPOTrainer keeps an internal frozen reference copy efficiently.
#
# NOTE: fast_inference=False because Granite 4.0 is a hybrid Mamba/Attention
# model (granitemoehybrid). Unsloth's vLLM→HF state_dict conversion doesn't
# handle Mamba layers yet, so we load directly via HuggingFace to avoid the
# UnboundLocalError in _get_vllm_state_dict.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_NEW,
    load_in_4bit   = LOAD_IN_4BIT,
    load_in_8bit   = LOAD_IN_FP8,
    fast_inference = False,           # vLLM path broken for granitemoehybrid
    max_lora_rank  = LORA_R,
)
tokenizer.pad_token = tokenizer.eos_token

print(f"Loaded: {MODEL_NAME}")
print(f"Dtype:  {next(model.parameters()).dtype}")

==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6. vLLM: 1.0.0.dev20260223+cu128.
   \\   /|    NVIDIA GeForce RTX 5070 Ti. Num GPUs = 1. Max memory: 15.466 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.0.dev20260223+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


[unsloth_zoo.log|WARNING]Unsloth: unsloth/smollm-135m-instruct-bnb-4bit can only handle sequence lengths of at most 2048.
But with kaiokendev's RoPE scaling of 4.0, it can be magically be extended to 8192!


Loaded: HuggingFaceTB/SmolLM-135M-Instruct
Dtype:  torch.bfloat16


In [10]:
# ── LoRA adapter ──────────────────────────────────────────────────────────────
# use_gradient_checkpointing="unsloth" asynchronously offloads activations to
# system RAM — only ~1% slower but saves an extra 30%+ VRAM.
# lora_dropout=0 and bias="none" are Unsloth-optimized defaults.
model = FastLanguageModel.get_peft_model(
    model,
    r                         = LORA_R,
    lora_alpha                = LORA_ALPHA,
    target_modules            = LORA_TARGET_MODULES,
    lora_dropout              = 0,           # 0 is optimized for Unsloth kernels
    bias                      = "none",      # "none" is optimized
    use_gradient_checkpointing = "unsloth",  # async offload → extra 30% VRAM saving
    random_state              = 42,
)
model.print_trainable_parameters()


Unsloth 2026.2.1 patched 30 layers with 30 QKV layers, 30 O layers and 30 MLP layers.


trainable params: 2,442,240 || all params: 136,957,248 || trainable%: 1.7832


In [11]:
# ── Sanity check: single greedy decode before training ───────────────────────
sample = eval_ds[0]
input_ids = tokenizer.apply_chat_template(
    sample["prompt"], add_generation_prompt=True, return_tensors="pt"
).to(DEVICE).repeat_interleave(G, dim=0)  # GRPO group size G

with torch.inference_mode():
    out = model.generate(
        input_ids        = input_ids,
        max_new_tokens   = 1024,
        do_sample        = True,
        pad_token_id     = tokenizer.eos_token_id,
    )

for i in range(G):
    response = tokenizer.decode(out[i, input_ids.shape[1]:], skip_special_tokens=True)
    gold     = sample["answer"]
    pre_r    = reward_fn([response], answer=[gold])[0]
    print(f"Sample {i+1} | Gold {gold!r} Reward {pre_r:.2f}:\n{response}\n{'─'*70}")


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Sample 1 | Gold '18' Reward 0.39:
Based on your understanding of math writing, here's how we represent the chicken's daily routine:

"Janet, I take a fresh chicken down at the farmers' market every day, just like I eat three for breakfast. The chicken I need for the day is $2 per fresh duck eggs. I need to make $0.35 per fresh duck egg. To make my way to the farmer's market, I also sell a 10% bit over half of the remaining duck eggs for muffins each day. I can sell 10 out of 15 pounds of fresh duck eggs for $0.35 per pound."

To solve this problem, you would need to know how much she eats in the first day (3 pounds and 587 calories) and how many times she takes away some of the money, which is $56.23 for the $2 per fresh duck eggs, plus the amount she sells for muffins for each 10% of the amount she eats. Therefore, the total amount eaten during the first day is 3 * $3.75 = $17.37.

After the farmer's market, you find that the chicken, or rather, the chicken ate most of the remaining d

In [ ]:
for i in range(G):
    response = tokenizer.decode(out[i, input_ids.shape[1]:], skip_special_tokens=True)
    gold     = sample["answer"]
    pre_r    = reward_fn([response], answer=[gold])[0]
    print(f"Sample {i+1} | Gold {gold!r} Reward {pre_r:.2f}:\n{response}\n{'─'*70}")

In [12]:
# ── GRPOTrainer ───────────────────────────────────────────────────────────────
# Key points:
#   • Generation uses HF model.generate() (vLLM disabled for this hybrid model)
#   • Unsloth memory-efficient linear kernels slice logit memory 8×+
#   • No separate ref_model load — GRPOTrainer holds a frozen copy internally
#   • optim="adamw_8bit" saves ~75% optimizer state VRAM
training_args = GRPOConfig(
    output_dir                  = "./grpo-lora",
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = 4,               # ↑ from 2 → 2× fewer steps
    gradient_accumulation_steps = 1,
    num_generations             = G,               # ↓ from 16 → G (4); biggest speedup
    max_completion_length       = 256,             # ↓ from 1024; GSM8K rarely needs >200 tokens
    temperature                 = TEMPERATURE,
    learning_rate               = LR,
    optim                       = "adamw_8bit",   # 8-bit Adam: ~75% less optimizer VRAM
    beta                        = 0.01,           # KL penalty coefficient
    epsilon                     = CLIP_EPS,
    logging_steps               = 1,
    save_strategy               = "no",
    report_to                   = "none",
)

trainer = GRPOTrainer(
    model         = model,
    tokenizer     = tokenizer,
    reward_funcs  = [reward_fn],
    args          = training_args,
    train_dataset = train_ds,
)

print("Starting GRPO training...")
trainer.train()
print("Training complete.")


Reward function defined.
Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 4 to the `num_generations` of 16
Starting GRPO training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 2,442,240 of 136,957,248 (1.78% trained)


Unsloth: Will smartly offload gradients to save VRAM!


KeyboardInterrupt: 

In [ ]:
# ── Post-training eval ────────────────────────────────────────────────────────
# for_inference() fuses the LoRA adapter weights → 2× faster native inference.
FastLanguageModel.for_inference(model)

total_reward = 0.0
for item in eval_ds:
    input_ids = tokenizer.apply_chat_template(
        item["prompt"], add_generation_prompt=True, return_tensors="pt"
    ).to(DEVICE)

    with torch.inference_mode():
        out = model.generate(
            input_ids      = input_ids,
            max_new_tokens = MAX_NEW,
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
        )

    completion = tokenizer.decode(out[0, input_ids.shape[1]:], skip_special_tokens=True)
    r = reward_fn([completion], answer=[item["answer"]])[0]
    total_reward += r
    print(f"  [r={r:.2f}] gold={item['answer']!r} | {completion[:80]!r}")

post_acc = total_reward / len(eval_ds)
print(f"\nPost-train mean reward: {post_acc:.4f}")
print(f"Pre-train reward (sanity cell): {pre_r:.4f}")


In [ ]:
# ── Save ──────────────────────────────────────────────────────────────────────
model.save_pretrained("./grpo-lora")
tokenizer.save_pretrained("./grpo-lora")
print("Saved LoRA adapter to ./grpo-lora")
